# LeetCode 1730 — Shortest Path to Get Food bằng Python

Link đề: https://leetcode.com/problems/shortest-path-to-get-food/ · Độ khó: **Medium** · Chủ đề: Array, Breadth-First Search, Matrix

> **Bài Premium.** Đề bài trong notebook được viết lại theo hiểu biết về bài (nguồn: bản mô tả của doocs/leetcode). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def getFood(self, grid: List[List[str]]) -> int:
        ...
```

Lộ trình: **DFS quay lui thử mọi đường đi đơn** (mũ, lưới nhỏ) → **BFS từ vị trí `*`** với mảng khoảng cách (`O(m·n)`) → **BFS theo tầng**, đánh dấu ngay trên lưới, dừng ở món ăn đầu tiên → **BFS đa nguồn** từ mọi ô `#` (góc nhìn ngược) → (mở rộng) **A\*** với heuristic Manhattan.

## 1. Mục tiêu học

1. Mô hình hóa lưới thành **đồ thị không trọng số**: mỗi ô đi được là một đỉnh, nối 4 hướng.
2. Hiểu vì sao **BFS** cho đường đi ngắn nhất trên đồ thị không trọng số: đỉnh được lấy ra theo thứ tự khoảng cách không giảm.
3. Cài BFS chuẩn với `deque`, đánh dấu **đã thăm khi đẩy vào hàng đợi** (không phải khi lấy ra) để tránh trùng lặp.
4. Biết **BFS đa nguồn** (multi-source BFS) và khi nào nên đảo chiều tìm kiếm.
5. Xử lý trường hợp **không tới được** (trả về `-1`) và kiểm chứng bằng vét cạn DFS trên lưới nhỏ.

## 2. Đề bài & ví dụ

Bạn đang đói và muốn ăn càng sớm càng tốt — tìm **đường đi ngắn nhất** tới **một ô thức ăn bất kỳ**.

Lưới ký tự `grid` kích thước `m × n` gồm các loại ô:
- `'*'` — vị trí của bạn. Có **đúng một** ô `'*'`.
- `'#'` — ô thức ăn. Có thể có **nhiều** ô.
- `'O'` — ô trống, đi qua được.
- `'X'` — chướng ngại, không đi qua được.

Mỗi bước đi sang ô kề **bắc/đông/nam/tây** nếu ô đó không phải chướng ngại. Trả về **độ dài** đường đi ngắn nhất tới ô thức ăn; nếu không có đường, trả về `-1`.

**Ví dụ 1:** → `3`.
```
X X X X X X
X * O O O X
X O O # O X
X X X X X X
```
**Ví dụ 2:** → `-1` (thức ăn bị chặn).
```
X X X X X
X * X O X
X O X # X
X X X X X
```
**Ví dụ 3:** → `6` (có nhiều ô thức ăn, ô ở dưới gần hơn).

**Ví dụ 4 (theo bản doocs):** lưới như ví dụ 3 nhưng hàng dưới cùng là `"OOOOOOOO"`, ghi đáp án `5`. **Lưu ý:** chạy BFS (và cả DFS vét cạn) trên đúng lưới này cho kết quả `6`, không phải `5` — hàng dưới cùng mở ra nhưng không tạo đường ngắn hơn. Rất có thể lưới của ví dụ 4 trong nguồn bị chép sai. Notebook dùng ví dụ 1–3 để kiểm thử và phân tích riêng ví dụ 4 ở cell dưới; **hãy đối chiếu lại đề gốc**.

**Ràng buộc:**
- `1 ≤ m, n ≤ 200`
- `grid[row][col]` là `'*'`, `'X'`, `'O'` hoặc `'#'`; có đúng một `'*'`.

In [1]:
import random
import time
import heapq
from collections import deque
from typing import List

def G(rows):
    """Tiện ích: chuyển list chuỗi thành lưới ký tự."""
    return [list(r) for r in rows]

examples = [
    (G(["XXXXXX", "X*OOOX", "XOO#OX", "XXXXXX"]), 3),
    (G(["XXXXX", "X*XOX", "XOX#X", "XXXXX"]), -1),
    (G(["XXXXXXXX", "X*OXO#OX", "XOOXOOXX", "XOOOO#OX", "XXXXXXXX"]), 6),
]
for g, want in examples:
    print("\n".join(" ".join(r) for r in g), f"\n-> kỳ vọng {want}\n")

# Ví dụ 4 theo bản doocs (ghi đáp án 5) — được phân tích ở mục 8 và 12
ex4 = G(["XXXXXXXX", "X*OXO#OX", "XOOXOOXX", "XOOOO#OX", "OOOOOOOO"])
print("ví dụ 4 (bản doocs, ghi đáp án 5):")
print("\n".join(" ".join(r) for r in ex4))

X X X X X X
X * O O O X
X O O # O X
X X X X X X 
-> kỳ vọng 3

X X X X X
X * X O X
X O X # X
X X X X X 
-> kỳ vọng -1

X X X X X X X X
X * O X O # O X
X O O X O O X X
X O O O O # O X
X X X X X X X X 
-> kỳ vọng 6

ví dụ 4 (bản doocs, ghi đáp án 5):
X X X X X X X X
X * O X O # O X
X O O X O O X X
X O O O O # O X
O O O O O O O O


## 3. Kiến thức nền

### 3.1 Duyệt 4 hướng và kiểm tra biên

In [2]:
DIRS = [(-1, 0), (0, 1), (1, 0), (0, -1)]     # bắc, đông, nam, tây

def neighbors(grid, r, c):
    m, n = len(grid), len(grid[0])
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != "X":
            yield nr, nc

g = examples[0][0]
print(list(neighbors(g, 1, 1)))

[(1, 2), (2, 1)]


### 3.2 Tìm vị trí bắt đầu

In [3]:
def find_start(grid):
    for r, row in enumerate(grid):
        for c, ch in enumerate(row):
            if ch == "*":
                return r, c

print(find_start(examples[2][0]))

(1, 1)


### 3.3 `deque` làm hàng đợi FIFO

`append` thêm vào cuối, `popleft` lấy ở đầu — cả hai `O(1)`. BFS dùng hàng đợi FIFO; DFS dùng stack (LIFO).

In [4]:
q = deque([(0, 0)])
q.append((0, 1)); q.append((1, 0))
print(q.popleft(), q)

(0, 0) deque([(0, 1), (1, 0)])


### 3.4 BFS cho khoảng cách ngắn nhất

Trên đồ thị **không trọng số**, BFS thăm các đỉnh theo **tầng**: tầng 0 là nguồn, tầng 1 là các đỉnh kề nguồn, … Đỉnh ở tầng `d` có khoảng cách ngắn nhất đúng bằng `d`. Đánh dấu đỉnh là "đã thăm" **ngay khi đẩy vào hàng đợi** — tránh đẩy một đỉnh nhiều lần.

In [5]:
def bfs_levels(grid, src):
    dist = {src: 0}; q = deque([src])
    while q:
        u = q.popleft()
        for v in neighbors(grid, *u):
            if v not in dist:
                dist[v] = dist[u] + 1; q.append(v)
    return dist

d = bfs_levels(examples[0][0], (1, 1))
for r in range(4):
    print(" ".join(f"{d.get((r, c), '.'):>2}" if examples[0][0][r][c] != "X" else " X" for c in range(6)))

 X  X  X  X  X  X
 X  0  1  2  3  X
 X  1  2  3  4  X
 X  X  X  X  X  X


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: từ `*` ở `(1,1)`, bản đồ khoảng cách (cell trên) cho thấy ô `#` ở `(2,3)` có khoảng cách `3` — ví dụ đường `(1,1) → (1,2) → (1,3) → (2,3)`.

Hình dung như **sóng lan**: ở bước 1 sóng chạm các ô kề, bước 2 chạm ô kề của chúng… Lần đầu sóng chạm một ô `#` chính là khoảng cách ngắn nhất tới thức ăn **gần nhất**. Không cần tính khoảng cách tới mọi ô `#`.

Ví dụ 2: sóng từ `(1,1)` chỉ lan được tới `(2,1)` rồi bị tường chặn ⇒ không bao giờ chạm `#` ⇒ `-1`.

## 5. Cách 1 — Brute Force / baseline: DFS quay lui mọi đường đi đơn

**Ý tưởng:** từ `*`, thử mọi đường đi **không lặp ô** (đánh dấu ô đang trên đường, gỡ khi quay lui). Mỗi khi chạm `#`, cập nhật độ dài tốt nhất. Đúng theo định nghĩa (đường ngắn nhất luôn là đường đơn).

In [6]:
def get_food_dfs(grid):
    m, n = len(grid), len(grid[0])
    sr, sc = find_start(grid)
    on_path = [[False] * n for _ in range(m)]
    best = [float("inf")]

    def dfs(r, c, d):
        if d >= best[0]:
            return                              # cắt tỉa: không thể tốt hơn
        if grid[r][c] == "#":
            best[0] = d
            return
        on_path[r][c] = True
        for nr, nc in neighbors(grid, r, c):
            if not on_path[nr][nc]:
                dfs(nr, nc, d + 1)
        on_path[r][c] = False

    dfs(sr, sc, 0)
    return -1 if best[0] == float("inf") else best[0]

for g, want in examples:
    print(get_food_dfs(g), "kỳ vọng", want)

3 kỳ vọng 3
-1 kỳ vọng -1
6 kỳ vọng 6


**Độ phức tạp:** số đường đi đơn trên lưới tăng theo hàm mũ (`O(3^{mn})` cận trên thô). Cắt tỉa `d ≥ best` giúp một phần nhưng worst case vẫn mũ — ví dụ lưới toàn `'O'` không có thức ăn.

**Nút thắt:** một ô được thăm qua rất nhiều đường khác nhau. BFS thăm mỗi ô **một lần**, với khoảng cách ngắn nhất.

In [7]:
for k in [3, 4, 5]:
    g = G(["*" + "O" * (k - 1)] + ["O" * k for _ in range(k - 1)])   # lưới k×k không có thức ăn
    t0 = time.perf_counter(); r = get_food_dfs(g)
    print(f"lưới {k}x{k} không có '#': {r}, {time.perf_counter() - t0:.3f}s")

lưới 3x3 không có '#': -1, 0.000s
lưới 4x4 không có '#': -1, 0.001s
lưới 5x5 không có '#': -1, 0.094s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đồ thị không trọng số ⇒ **BFS** từ `*`. Khi lấy ra (hoặc khi đẩy vào) một ô `#`, khoảng cách của nó là đáp án — vì BFS xét các ô theo khoảng cách không giảm, ô `#` đầu tiên gặp là ô **gần nhất**.

**Invariant:** hàng đợi luôn chứa các ô có khoảng cách `d` rồi `d + 1` (không quá hai tầng liên tiếp); `dist[v]` của mọi ô đã đánh dấu là khoảng cách ngắn nhất.

**Quy tắc:**
- Nếu ô kề là `'X'` hoặc ngoài lưới ⇒ bỏ qua.
- Nếu ô kề đã thăm ⇒ bỏ qua (đã có khoảng cách ≤).
- Nếu ô kề là `'#'` ⇒ trả về `d + 1` ngay (kiểm tra khi đẩy, dừng sớm hơn một tầng).
- Hàng đợi rỗng mà chưa gặp `'#'` ⇒ `-1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS với mảng khoảng cách

In [8]:
def get_food_bfs(grid):
    m, n = len(grid), len(grid[0])
    sr, sc = find_start(grid)
    dist = [[-1] * n for _ in range(m)]
    dist[sr][sc] = 0
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "#":
            return dist[r][c]
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != "X" and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return -1

for g, want in examples:
    print(get_food_bfs(g), "kỳ vọng", want)

3 kỳ vọng 3
-1 kỳ vọng -1
6 kỳ vọng 6


**Độ phức tạp:** mỗi ô vào hàng đợi tối đa 1 lần, mỗi ô xét 4 láng giềng ⇒ Time `O(m·n)`, Space `O(m·n)`.

### 7.2 Cách 3 — BFS theo tầng, đánh dấu trên bản sao lưới, kiểm tra `#` khi đẩy

Không cần mảng `dist`: đếm tầng bằng biến `steps`, xử lý hết `len(q)` phần tử của tầng hiện tại. Đánh dấu đã thăm bằng cách ghi `'X'` vào **bản sao** lưới (không phá input của người gọi).

In [9]:
def get_food_levels(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    sr, sc = find_start(g)
    g[sr][sc] = "X"
    q = deque([(sr, sc)])
    steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    ch = g[nr][nc]
                    if ch == "#":
                        return steps
                    if ch == "O":
                        g[nr][nc] = "X"
                        q.append((nr, nc))
    return -1

for g_, want in examples:
    print(get_food_levels(g_), "kỳ vọng", want)

3 kỳ vọng 3
-1 kỳ vọng -1
6 kỳ vọng 6


**Độ phức tạp:** Time `O(m·n)`, Space `O(m·n)` (bản sao + hàng đợi). Kiểm tra `#` khi đẩy giúp dừng sớm hơn một tầng so với kiểm tra khi lấy ra.

### 7.3 Cách 4 — BFS đa nguồn từ mọi ô `#`

Đảo chiều: đẩy **tất cả** ô `#` vào hàng đợi với khoảng cách 0, BFS cho tới khi chạm `*`. Khoảng cách tại `*` = khoảng cách tới ô thức ăn **gần nhất** (vì đồ thị vô hướng). Hữu ích khi có nhiều truy vấn "từ ô `x`, thức ăn gần nhất cách bao xa?" — một lần BFS đa nguồn trả lời cho mọi ô.

In [10]:
def get_food_multisource(grid):
    m, n = len(grid), len(grid[0])
    dist = [[-1] * n for _ in range(m)]
    q = deque()
    for r in range(m):
        for c in range(n):
            if grid[r][c] == "#":
                dist[r][c] = 0
                q.append((r, c))
    while q:
        r, c = q.popleft()
        if grid[r][c] == "*":
            return dist[r][c]
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != "X" and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return -1

for g, want in examples:
    print(get_food_multisource(g), "kỳ vọng", want)

3 kỳ vọng 3
-1 kỳ vọng -1
6 kỳ vọng 6


**Độ phức tạp:** Time `O(m·n)`, Space `O(m·n)`.

### 7.4 Cách 5 (mở rộng) — A\* với heuristic "khoảng cách Manhattan tới ô thức ăn gần nhất"

A\* ưu tiên ô có `g + h` nhỏ nhất (`g` = số bước đã đi, `h` = ước lượng còn lại). Heuristic `h = min Manhattan tới các ô #` là **chấp nhận được** (không vượt khoảng cách thật) ⇒ A\* cho đáp án đúng. Với nhiều ô `#`, tính `h` tốn `O(F)` mỗi ô — chỉ có lợi khi thức ăn ít và lưới rộng, mở.

In [11]:
def get_food_astar(grid):
    m, n = len(grid), len(grid[0])
    foods = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == "#"]
    if not foods:
        return -1
    def h(r, c):
        return min(abs(r - fr) + abs(c - fc) for fr, fc in foods)
    sr, sc = find_start(grid)
    best_g = {(sr, sc): 0}
    pq = [(h(sr, sc), 0, sr, sc)]
    while pq:
        f, g, r, c = heapq.heappop(pq)
        if g > best_g.get((r, c), float("inf")):
            continue
        if grid[r][c] == "#":
            return g
        for nr, nc in neighbors(grid, r, c):
            ng = g + 1
            if ng < best_g.get((nr, nc), float("inf")):
                best_g[(nr, nc)] = ng
                heapq.heappush(pq, (ng + h(nr, nc), ng, nr, nc))
    return -1

for g_, want in examples:
    print(get_food_astar(g_), "kỳ vọng", want)

3 kỳ vọng 3
-1 kỳ vọng -1
6 kỳ vọng 6


**Độ phức tạp:** worst case `O(m·n · (log(mn) + F))` với `F` số ô thức ăn; tốt nhất thăm rất ít ô.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 3 — in hàng đợi theo từng tầng và lưới đánh dấu:

In [12]:
def trace_levels(grid):
    g = [row[:] for row in grid]; m, n = len(g), len(g[0])
    sr, sc = find_start(g); g[sr][sc] = "X"; q = deque([(sr, sc)]); steps = 0
    print(f"tầng 0: {list(q)}")
    while q:
        steps += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n:
                    if g[nr][nc] == "#":
                        print(f"tầng {steps}: chạm thức ăn tại {(nr, nc)} từ {(r, c)} -> trả về {steps}")
                        return steps
                    if g[nr][nc] == "O":
                        g[nr][nc] = str(steps % 10); q.append((nr, nc))
        print(f"tầng {steps}: {list(q)}")
    print("hết hàng đợi -> -1")
    return -1

trace_levels(examples[2][0])

tầng 0: [(1, 1)]
tầng 1: [(1, 2), (2, 1)]
tầng 2: [(2, 2), (3, 1)]
tầng 3: [(3, 2)]
tầng 4: [(3, 3)]
tầng 5: [(3, 4)]
tầng 6: chạm thức ăn tại (3, 5) từ (3, 4) -> trả về 6


6

In [13]:
def show_distance_map(grid):
    m, n = len(grid), len(grid[0]); sr, sc = find_start(grid)
    dist = [[None] * n for _ in range(m)]; dist[sr][sc] = 0; q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(grid, r, c):
            if dist[nr][nc] is None:
                dist[nr][nc] = dist[r][c] + 1; q.append((nr, nc))
    for r in range(m):
        print(" ".join(" X" if grid[r][c] == "X" else (f"#{dist[r][c]}" if grid[r][c] == "#" and dist[r][c] is not None
                       else (f"{dist[r][c]:>2}" if dist[r][c] is not None else " ?")) for c in range(n)))

print("bản đồ khoảng cách ví dụ 3 (#d = ô thức ăn ở khoảng cách d):")
show_distance_map(examples[2][0])
print("\nví dụ 4 (bản doocs):")
show_distance_map(ex4)

bản đồ khoảng cách ví dụ 3 (#d = ô thức ăn ở khoảng cách d):
 X  X  X  X  X  X  X  X
 X  0  1  X  7 #8  9  X
 X  1  2  X  6  7  X  X
 X  2  3  4  5 #6  7  X
 X  X  X  X  X  X  X  X

ví dụ 4 (bản doocs):
 X  X  X  X  X  X  X  X
 X  0  1  X  7 #8  9  X
 X  1  2  X  6  7  X  X
 X  2  3  4  5 #6  7  X
 4  3  4  5  6  7  8  9


Ở ví dụ 3, ô `#` phía trên `(1,5)` bị tường cột 3 chắn đường trực tiếp nên phải đi vòng xa (khoảng cách 8), còn ô `#` phía dưới `(3,5)` có khoảng cách 6 ⇒ đáp án 6. Ở ví dụ 4 (bản doocs), hàng dưới cùng mở ra nhưng không tạo đường ngắn hơn: bản đồ cho thấy ô `#` gần nhất vẫn ở khoảng cách 6 — khớp cận dưới Manhattan, mâu thuẫn với đáp án `5` ghi trong nguồn (đối chiếu lại đề gốc).

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def getFood(self, grid: List[List[str]]) -> int:
        m, n = len(grid), len(grid[0])
        start = next((r, c) for r in range(m) for c in range(n) if grid[r][c] == "*")
        seen = [[False] * n for _ in range(m)]
        seen[start[0]][start[1]] = True
        q = deque([start])
        steps = 0
        while q:
            steps += 1
            for _ in range(len(q)):
                r, c = q.popleft()
                for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc]:
                        ch = grid[nr][nc]
                        if ch == "#":
                            return steps
                        if ch == "O":
                            seen[nr][nc] = True
                            q.append((nr, nc))
        return -1

sol = Solution()
for g, want in examples:
    got = sol.getFood(g)
    print(got, "OK" if got == want else "SAI")

3 OK
-1 OK
6 OK


## 10. Vì sao thuật toán đúng

**BFS cho khoảng cách ngắn nhất.** Chứng minh quy nạp theo tầng: mọi ô được đẩy vào hàng đợi ở tầng `d` có khoảng cách ngắn nhất từ `*` đúng bằng `d`.
- Tầng 0: chỉ có `*`, khoảng cách 0. ✓
- Giả sử đúng tới tầng `d`. Một ô `v` được đẩy ở tầng `d + 1` khi nó kề một ô `u` tầng `d` và chưa được đánh dấu. Có đường dài `d + 1` tới `v` (qua `u`). Nếu có đường ngắn hơn `≤ d`, ô kề cuối cùng trên đường đó có khoảng cách `≤ d − 1` ⇒ đã được xử lý ở tầng `≤ d − 1` ⇒ `v` đã được đẩy ở tầng `≤ d` — mâu thuẫn với việc `v` chưa đánh dấu. ✓

**Dừng ở ô `#` đầu tiên là đúng.** Các tầng được xử lý theo thứ tự tăng ⇒ ô `#` đầu tiên được phát hiện thuộc tầng nhỏ nhất có chứa thức ăn ⇒ là khoảng cách tới thức ăn gần nhất. Kiểm tra `#` khi đẩy (tầng `steps`) tương đương kiểm tra khi lấy ra, chỉ sớm hơn.

**Không đi xuyên qua `#`?** Thật ra đi qua `#` cũng không bao giờ cần: nếu đường ngắn nhất tới thức ăn đi qua một ô `#` khác, thì ô `#` đó còn gần hơn. Vì vậy không đẩy `#` vào hàng đợi là an toàn.

**`-1`** khi hàng đợi cạn: đã thăm toàn bộ thành phần liên thông chứa `*` mà không có `#`.

**BFS đa nguồn** đúng vì khoảng cách trên đồ thị vô hướng đối xứng: `min_f dist(*, f) = min_f dist(f, *)`, và BFS đa nguồn tương đương BFS từ một siêu đỉnh nối tới mọi `#`.

## 11. Phân tích Time / Space

Gọi `m, n ≤ 200` ⇒ tối đa `4·10⁴` ô.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. DFS mọi đường | mũ | `O(mn)` | lưới tí hon |
| 2. BFS + dist | `O(mn)` | `O(mn)` | chuẩn |
| 3. BFS theo tầng | `O(mn)` | `O(mn)` | **nộp bài** (dừng sớm) |
| 4. BFS đa nguồn | `O(mn)` | `O(mn)` | nhiều truy vấn |
| 5. A\* | `O(mn (log mn + F))` worst | `O(mn)` | lưới mở, ít thức ăn |

In [15]:
def random_grid(m, n, p_wall, p_food, rng):
    g = [["X" if rng.random() < p_wall else ("#" if rng.random() < p_food else "O") for _ in range(n)] for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    g[r][c] = "*"
    return g

rng = random.Random(1)
big = random_grid(200, 200, 0.3, 0.0005, rng)
for name, fn in [("bfs", get_food_bfs), ("levels", get_food_levels), ("multi", get_food_multisource),
                 ("astar", get_food_astar), ("submit", Solution().getFood)]:
    t0 = time.perf_counter(); r = fn(big)
    print(f"{name:6s}: {r:4d}  {time.perf_counter() - t0:.3f}s")

bfs   :   50  0.002s
levels:   50  0.002s
multi :   50  0.011s
astar :   50  0.002s
submit:   50  0.002s


## 12. Bộ test

In [16]:
fns = [get_food_bfs, get_food_levels, get_food_multisource, get_food_astar, lambda g: Solution().getFood(g)]

for g, want in examples:
    assert get_food_dfs(g) == want
    for fn in fns:
        assert fn(g) == want

edge = [
    (G(["*"]), -1),                   # 1 ô, không thức ăn
    (G(["*#"]), 1),
    (G(["#*#"]), 1),
    (G(["*X#"]), -1),
    (G(["*", "O", "#"]), 2),
    (G(["*O", "X#"]), 2),
    (G(["*OOO", "XXXO", "#OOO"]), 8),  # phải đi vòng: 3 phải, 2 xuống, 3 trái
]
for g, want in edge:
    assert get_food_dfs(g) == want, g
    for fn in fns:
        assert fn(g) == want, (fn, g)

rng = random.Random(2024)
for _ in range(500):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    g = random_grid(m, n, rng.choice([0.1, 0.3, 0.5]), rng.choice([0.05, 0.2]), rng)
    want = get_food_dfs(g)
    for fn in fns:
        assert fn(g) == want, (fn, g)
print("ví dụ + edge + 500 lưới ngẫu nhiên khớp DFS vét cạn")
print("ví dụ 4 (bản doocs): DFS vét cạn =", get_food_dfs(ex4), "| BFS =", get_food_bfs(ex4), "(nguồn ghi 5 -> đối chiếu lại đề gốc)")

ví dụ + edge + 500 lưới ngẫu nhiên khớp DFS vét cạn
ví dụ 4 (bản doocs): DFS vét cạn = 6 | BFS = 6 (nguồn ghi 5 -> đối chiếu lại đề gốc)


In [17]:
# lưới lớn hơn: các cách BFS khớp nhau; input không bị sửa
rng = random.Random(7)
for _ in range(50):
    m, n = rng.randint(1, 60), rng.randint(1, 60)
    g = random_grid(m, n, rng.random() * 0.5, rng.random() * 0.02, rng)
    snapshot = [row[:] for row in g]
    r = get_food_bfs(g)
    assert r == get_food_levels(g) == get_food_multisource(g) == get_food_astar(g) == Solution().getFood(g)
    assert g == snapshot
print("lưới tới 60x60: năm cách khớp nhau, lưới gốc không bị thay đổi")

lưới tới 60x60: năm cách khớp nhau, lưới gốc không bị thay đổi


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng DFS (stack) và trả về độ dài đường **đầu tiên** tìm được

DFS không thăm theo thứ tự khoảng cách ⇒ đường đầu tiên chạm `#` chưa chắc ngắn nhất.

In [18]:
def wrong_dfs_first(grid):
    m, n = len(grid), len(grid[0]); s = find_start(grid)
    seen = {s}; st = [(s, 0)]
    while st:
        (r, c), d = st.pop()
        if grid[r][c] == "#":
            return d                           # SAI: đường đầu tiên của DFS
        for v in neighbors(grid, r, c):
            if v not in seen:
                seen.add(v); st.append((v, d + 1))
    return -1

rng = random.Random(4)
for _ in range(5000):
    g = random_grid(rng.randint(2, 5), rng.randint(2, 5), 0.2, 0.15, rng)
    if wrong_dfs_first(g) != Solution().getFood(g):
        print("\n".join("".join(r) for r in g))
        print("sai =", wrong_dfs_first(g), "| đúng =", Solution().getFood(g))
        break

*#O
O##
OOO
OOO
OXO
sai = 7 | đúng = 1


**Sửa:** BFS với hàng đợi FIFO.

### Lỗi 2 — Đánh dấu đã thăm khi **lấy ra** thay vì khi **đẩy vào**

Vẫn cho kết quả đúng nhưng một ô có thể bị đẩy vào hàng đợi nhiều lần (từ nhiều láng giềng) ⇒ chậm hơn nhiều, có thể TLE/MLE.

In [19]:
def bfs_mark_on_pop(grid):
    m, n = len(grid), len(grid[0]); s = find_start(grid)
    seen = set(); q = deque([(s, 0)]); pushes = 0
    while q:
        (r, c), d = q.popleft()
        if (r, c) in seen:
            continue
        seen.add((r, c))                       # đánh dấu khi lấy ra
        if grid[r][c] == "#":
            return d, pushes
        for v in neighbors(grid, r, c):
            if v not in seen:
                q.append((v, d + 1)); pushes += 1
    return -1, pushes

g = G(["*" + "O" * 59] + ["O" * 60 for _ in range(58)] + ["O" * 59 + "#"])
res, pushes = bfs_mark_on_pop(g)
print(f"đánh dấu khi lấy ra: kết quả {res}, số lần đẩy = {pushes} (số ô = {60 * 60})")

đánh dấu khi lấy ra: kết quả 118, số lần đẩy = 7080 (số ô = 3600)


**Sửa:** đánh dấu ngay khi đẩy (Cách 2, 3) ⇒ mỗi ô đẩy tối đa 1 lần.

### Lỗi 3 — Coi `'#'` là chướng ngại hoặc quên rằng `'*'` không phải ô trống

Điều kiện `if ch == 'O'` mà không kiểm tra `'#'` trước ⇒ không bao giờ "đi vào" ô thức ăn ⇒ luôn `-1`.

In [20]:
def wrong_only_O(grid):
    m, n = len(grid), len(grid[0]); s = find_start(grid)
    seen = {s}; q = deque([(s, 0)])
    while q:
        (r, c), d = q.popleft()
        if grid[r][c] == "#":
            return d
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen and grid[nr][nc] == "O":   # SAI: bỏ qua '#'
                seen.add((nr, nc)); q.append(((nr, nc), d + 1))
    return -1

for g, want in examples[:1]:
    print("sai =", wrong_only_O(g), "| đúng =", want)

sai = -1 | đúng = 3


**Sửa:** xử lý `'#'` (trả về) trước, rồi mới đến `'O'` (đẩy vào).

### Lỗi 4 — Đếm tầng sai: tăng `steps` sau khi kiểm tra, hoặc trả về `steps` khi kiểm tra lúc lấy ra

Trộn hai quy ước "kiểm tra khi đẩy" và "kiểm tra khi lấy ra" làm lệch một bước.

In [21]:
def wrong_steps(grid):
    m, n = len(grid), len(grid[0]); s = find_start(grid)
    seen = {s}; q = deque([s]); steps = 0
    while q:
        for _ in range(len(q)):
            r, c = q.popleft()
            if grid[r][c] == "#":
                return steps + 1               # SAI: kiểm tra khi lấy ra thì phải trả về steps
            for v in neighbors(grid, r, c):
                if v not in seen:
                    seen.add(v); q.append(v)
        steps += 1
    return -1

for g, want in examples:
    print("sai =", wrong_steps(g), "| đúng =", want)

sai = 4 | đúng = 3
sai = -1 | đúng = -1
sai = 7 | đúng = 6


**Sửa:** chọn một quy ước và nhất quán: kiểm tra khi lấy ra ⇒ trả về số tầng hiện tại (đã tính tới ô đó); kiểm tra khi đẩy ⇒ trả về tầng của ô được đẩy.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường đi **ngắn nhất** trên **lưới**, mỗi bước chi phí bằng nhau" ⇒ **BFS**.
- Nhiều đích ⇒ BFS từ nguồn, dừng ở đích đầu tiên; nhiều nguồn ⇒ **BFS đa nguồn**.
- Đánh dấu khi đẩy; kiểm tra đích khi đẩy để dừng sớm.
- Chi phí khác nhau ⇒ Dijkstra; chi phí 0/1 ⇒ **0-1 BFS** (deque, đẩy đầu cho cạnh 0).
- Họ bài: 1091 *Shortest Path in Binary Matrix* (8 hướng), 542 *01 Matrix* (đa nguồn), 994 *Rotting Oranges* (đa nguồn theo tầng), 1926 *Nearest Exit from Entrance in Maze* (gần như y hệt).

## 15. Biến thể của bài

1. **Trả về đường đi cụ thể:** lưu `parent` của mỗi ô khi đẩy, truy ngược từ ô `#` tìm được.
2. **Khoảng cách tới thức ăn gần nhất cho MỌI ô** (542-style): BFS đa nguồn từ mọi `#`.
3. **Có thể phá tối đa `k` chướng ngại** (1293): trạng thái `(r, c, k_còn_lại)`.
4. **Ô có chi phí khác nhau** (ví dụ bùn tốn 2 bước): Dijkstra.
5. **Di chuyển 8 hướng:** thêm 4 hướng chéo vào `DIRS`.

In [22]:
def get_food_path(grid):
    """Biến thể 1: trả về (độ dài, danh sách ô trên đường đi)."""
    m, n = len(grid), len(grid[0]); s = find_start(grid)
    parent = {s: None}; q = deque([s])
    while q:
        u = q.popleft()
        if grid[u[0]][u[1]] == "#":
            path = []
            while u is not None:
                path.append(u); u = parent[u]
            return len(path) - 1, path[::-1]
        for v in neighbors(grid, *u):
            if v not in parent:
                parent[v] = u; q.append(v)
    return -1, []

for g, want in examples:
    d, path = get_food_path(g)
    assert d == want
    print(d, path)

3 [(1, 1), (1, 2), (1, 3), (2, 3)]
-1 []
6 [(1, 1), (1, 2), (2, 2), (3, 2), (3, 3), (3, 4), (3, 5)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. DFS quay lui | thử mọi đường đơn | mũ | `O(mn)` | kiểm chứng |
| 2. BFS + dist | sóng lan, mảng khoảng cách | `O(mn)` | `O(mn)` | khi cần khoảng cách mọi ô |
| 3. BFS theo tầng | đếm tầng, dừng sớm | `O(mn)` | `O(mn)` | **nộp bài** |
| 4. BFS đa nguồn | từ mọi `#` | `O(mn)` | `O(mn)` | nhiều điểm xuất phát |
| 5. A\* | ưu tiên `g + h` | thường nhỏ | `O(mn)` | lưới lớn, mở |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `getFood([["*","O","#"],["X","X","#"]])`?

**Bài 2 (tính tay):** vẽ bản đồ khoảng cách BFS cho `["*OX", "OOX", "X#O"]`.

**Bài 3 (điền TODO):** hoàn thiện `nearest_food_all(grid)` trả về ma trận khoảng cách tới ô `#` gần nhất cho mọi ô đi được (`-1` nếu không tới được), bằng BFS đa nguồn.

**Bài 4 (tự viết lại):** viết `get_food_8dir(grid)` cho phép đi 8 hướng (kể cả chéo).

**Bài 5 (mini challenge):** biến thể **được phá tối đa `k` ô `X`** trên đường đi. Viết BFS trạng thái `(r, c, còn_phá)` và kiểm chứng bằng cách thử mọi tập ô `X` bị phá (≤ `k` ô) trên lưới nhỏ. (Biến thể tự đặt, giống LeetCode 1293.)

In [23]:
def nearest_food_all(grid):
    # TODO: đẩy mọi '#' vào hàng đợi với khoảng cách 0, BFS
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `*` ở `(0,0)`. (2) `(0,1)` cách 1, `(0,2)` là `#` cách 2. (3) Đáp án `2`.

**Bài 2:** (1) `(0,0)=0`, `(0,1)=1`, `(1,0)=1`. (2) `(1,1)=2`. (3) `#` ở `(2,1)` cách 3; `(2,2)` cách 4 (đi qua `#`? — trong bài gốc không cần, nhưng bản đồ khoảng cách thuần đi qua được).

**Bài 3:** (1) `dist = [[-1]*n ...]`, mọi `#` có `dist = 0`. (2) BFS như Cách 4 nhưng không dừng ở `*`. (3) Ô `X` giữ `-1`.

**Bài 4:** (1) Thêm `(±1, ±1)` vào danh sách hướng. (2) Phần còn lại giữ nguyên. (3) So sánh với lưới mà đường chéo tạo đường tắt.

**Bài 5:** (1) `seen[r][c][k]`. (2) Bước vào ô `X` khi `k > 0` ⇒ trạng thái `k − 1`. (3) Kiểm chứng: với mỗi tập con ô `X` kích thước ≤ `k`, đổi thành `O` và chạy BFS thường; lấy min.

## 19. Đáp án tham khảo

In [24]:
from itertools import combinations

# Bài 1
assert Solution().getFood([["*", "O", "#"], ["X", "X", "#"]]) == 2

# Bài 2
assert Solution().getFood(G(["*OX", "OOX", "X#O"])) == 3

# Bài 3
def nearest_food_all_ans(grid):
    m, n = len(grid), len(grid[0]); dist = [[-1] * n for _ in range(m)]; q = deque()
    for r in range(m):
        for c in range(n):
            if grid[r][c] == "#":
                dist[r][c] = 0; q.append((r, c))
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(grid, r, c):
            if dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1; q.append((nr, nc))
    return dist

for g, want in examples:
    sr, sc = find_start(g)
    assert nearest_food_all_ans(g)[sr][sc] == want

# Bài 4
DIRS8 = DIRS + [(-1, -1), (-1, 1), (1, -1), (1, 1)]
def get_food_8dir(grid):
    m, n = len(grid), len(grid[0]); s = find_start(grid)
    seen = {s}; q = deque([s]); steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in DIRS8:
                nr, nc = r + dr, c + dc
                if 0 <= nr < m and 0 <= nc < n and (nr, nc) not in seen and grid[nr][nc] != "X":
                    if grid[nr][nc] == "#":
                        return steps
                    seen.add((nr, nc)); q.append((nr, nc))
    return -1

assert get_food_8dir(G(["*X", "X#"])) == 1 and Solution().getFood(G(["*X", "X#"])) == -1
print("Bài 1-4: OK")

Bài 1-4: OK


In [25]:
# Bài 5 — được phá tối đa k chướng ngại
def get_food_break(grid, k):
    m, n = len(grid), len(grid[0]); s = find_start(grid)
    seen = {(s[0], s[1], k)}; q = deque([(s[0], s[1], k)]); steps = 0
    while q:
        steps += 1
        for _ in range(len(q)):
            r, c, left = q.popleft()
            for dr, dc in DIRS:
                nr, nc = r + dr, c + dc
                if not (0 <= nr < m and 0 <= nc < n):
                    continue
                ch = grid[nr][nc]
                if ch == "#":
                    return steps
                nl = left - (ch == "X")
                if nl >= 0 and (nr, nc, nl) not in seen and ch != "*":
                    seen.add((nr, nc, nl)); q.append((nr, nc, nl))
    return -1

def brute_break(grid, k):
    walls = [(r, c) for r in range(len(grid)) for c in range(len(grid[0])) if grid[r][c] == "X"]
    best = -1
    for t in range(0, min(k, len(walls)) + 1):
        for subset in combinations(walls, t):
            g = [row[:] for row in grid]
            for r, c in subset:
                g[r][c] = "O"
            d = get_food_bfs(g)
            if d != -1 and (best == -1 or d < best):
                best = d
    return best

rng = random.Random(55)
for _ in range(300):
    g = random_grid(rng.randint(1, 4), rng.randint(1, 4), 0.4, 0.15, rng)
    k = rng.randint(0, 2)
    assert get_food_break(g, k) == brute_break(g, k), (g, k)
print("Bài 5: khớp vét cạn; ví dụ 2 với k=1 ->", get_food_break(examples[1][0], 1))

Bài 5: khớp vét cạn; ví dụ 2 với k=1 -> 3


## 20. Template nhớ nhanh + Checklist

```python
# BFS theo tầng trên lưới, dừng ở đích đầu tiên
seen[start] = True; q = deque([start]); steps = 0
while q:
    steps += 1
    for _ in range(len(q)):
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if in_bounds(nr, nc) and not seen[nr][nc]:
                if grid[nr][nc] == "#": return steps     # kiểm tra đích khi đẩy
                if grid[nr][nc] == "O":
                    seen[nr][nc] = True; q.append((nr, nc))
return -1
```

**Checklist trước khi submit:**
- [ ] BFS (hàng đợi FIFO), không phải DFS.
- [ ] Đánh dấu đã thăm **khi đẩy**.
- [ ] Xử lý `'#'` trước `'O'`; không đi qua `'X'`.
- [ ] Quy ước đếm tầng nhất quán (kiểm tra khi đẩy ⇒ trả về `steps`).
- [ ] Trả về `-1` khi hàng đợi cạn.
- [ ] Không sửa lưới input (hoặc sửa trên bản sao).